# Module 09 — YOLO Family (SOLUTIONS)

In [ ]:
import torch
import torch.nn.functional as F

def yolov1_loss(predictions, targets, S=7, B=2, C=20, lambda_coord=5.0, lambda_noobj=0.5):
    """
    YOLOv1 loss function.
    predictions: (batch, S, S, B*5+C)
    targets:     (batch, S, S, B*5+C)  — 0-padded for non-object cells
    """
    batch = predictions.shape[0]
    pred = predictions.view(batch, S, S, B, 5 + C)
    tgt  = targets.view(batch, S, S, B, 5 + C)

    obj_mask  = tgt[..., 4] == 1   # cells containing an object
    noobj_mask= tgt[..., 4] == 0

    # 1. Coordinate loss (only for responsible cells)
    xy_loss = F.mse_loss(pred[obj_mask][..., :2], tgt[obj_mask][..., :2], reduction='sum')
    # sqrt(w), sqrt(h) to make loss scale-invariant
    wh_loss = F.mse_loss(
        torch.sign(pred[obj_mask][..., 2:4]) * torch.sqrt(pred[obj_mask][..., 2:4].abs() + 1e-6),
        torch.sqrt(tgt[obj_mask][..., 2:4].abs() + 1e-6),
        reduction='sum'
    )
    coord_loss = lambda_coord * (xy_loss + wh_loss)

    # 2. Confidence loss
    obj_conf_loss   = F.mse_loss(pred[obj_mask][..., 4], tgt[obj_mask][..., 4], reduction='sum')
    noobj_conf_loss = lambda_noobj * F.mse_loss(pred[noobj_mask][..., 4], tgt[noobj_mask][..., 4], reduction='sum')

    # 3. Class loss
    class_loss = F.mse_loss(pred[obj_mask][..., 5:], tgt[obj_mask][..., 5:], reduction='sum')

    total = (coord_loss + obj_conf_loss + noobj_conf_loss + class_loss) / batch
    return total

# Test
pred = torch.randn(2, 7, 7, 2*5+20)
tgt  = torch.zeros_like(pred)
tgt[0, 3, 3, 0] = 1  # object in cell (3,3), box 0
loss = yolov1_loss(pred, tgt)
print(f'Loss: {loss.item():.4f}')